# AI Functions: Extensions & Cost Control
*Co-authored with CoCo*

The base lab (`cx-ai-functions-lab.ipynb`) built a CX telemetry pipeline with built-in AI Functions and a custom AI Function Studio router. This companion notebook shows where those functions **plug into the rest of the Cortex stack** — the semantic view, Cortex Analyst, Cortex Search, and a Cortex Agent — and how to keep spend under control at scale.

**Part 1 - Extensions** (how an AI-function UDF is reused elsewhere):
1. As a **custom tool for a Cortex Agent** - the agent's LLM calls your UDF via tool-use.
2. **Inside Cortex Analyst** - the UDF becomes a computed column the generated SQL can reference, alongside the governed semantic view.
3. In a **Cortex Search enrichment pipeline** - `AI_EXTRACT` / `AI_EMBED` process documents before indexing.

**Part 2 - Best practices & cost/spike prevention**: prompt engineering, model selection, monthly spend alerts, per-user limits, runaway-query cancellation, query tagging, and access control.

> `setup.sql` creates AND fully populates the real semantic view (`ANALYTICS.CX_ANALYTICS_SV`), Cortex Search service (`ANALYTICS.CHAT_SEARCH`), and agent (`ANALYTICS.CX_INTELLIGENCE_AGENT`) — so Extensions 1-3 **run live** here with no dependency on the base lab's run order (`thumbs_down_rate`, search hits, and the agent are all ready straight from setup). (This notebook folds in what used to be the standalone Conversational-BI lab.) The narrated YAML/DDL snippets show how each object is *wired*; the cell beneath each one queries the real object.

In [ ]:
# Connect using the active Snowsight session and pin role + database/schema/warehouse.
# SQL cells share this session, so the role set here applies to the CREATE cells below.
# Use a role that has CREATE FUNCTION/VIEW/TABLE on the schema AND the
# SNOWFLAKE.CORTEX_USER database role (ACCOUNTADMIN here; swap for your own).
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_role("ACCOUNTADMIN")
session.use_database("FIELD_CX_DEMO")
session.use_schema("AI_FUNCTIONS")
session.use_warehouse("CX_AI_FUNCTIONS_WH")
print("Context:", session.get_current_role(), session.get_current_database(),
      session.get_current_schema(), session.get_current_warehouse())

---
## Extension 1 - A custom AI-function UDF as an Agent tool

A Cortex Agent orchestrates tools to answer a question. Alongside Cortex Analyst (structured) and Cortex Search (unstructured), you can register **any SQL function** as a **generic custom tool**. When the user's question needs it, the agent's LLM emits a tool call, Snowflake executes your UDF, and the result flows back into the answer.

We first wrap an `AI_COMPLETE` call as a plain SQL UDF (`CLASSIFY_ESCALATION`). This is the same idea as the Studio-built `ROUTE_ESCALATION` from the base lab, but self-contained so this notebook runs on its own.

In [ ]:
%%sql -r create_escalation_udf
-- Self-contained AI-function UDF: label a message LOW / MEDIUM / HIGH.
-- Bounded, one-word prompt = predictable output length = predictable cost.
CREATE OR REPLACE FUNCTION CLASSIFY_ESCALATION(msg STRING)
RETURNS STRING
AS
$$
  UPPER(TRIM(AI_COMPLETE('llama3.1-8b',
    'You are an escalation router. Respond with EXACTLY one word and nothing else: '
    || 'LOW, MEDIUM, or HIGH. Rate the escalation urgency of this customer message.'
    || CHR(10) || 'Message: ' || msg)))
$$;

In [ ]:
%%sql -r escalation_sample
-- The UDF behaves like any scalar function - callable from SQL, an agent, or Analyst.
SELECT thread_id, CLASSIFY_ESCALATION(transcript) AS escalation_level
FROM CHAT_THREADS
LIMIT 5;

### Register the UDF as a generic tool in the agent spec

In the agent's specification you declare a **generic tool** and point it at the UDF. The `description` is what the agent's LLM reads to decide *when* to call it, so make it specific.

```yaml
tools:
  - tool_spec:
      type: generic
      name: classify_escalation
      description: >
        Returns the escalation urgency (LOW, MEDIUM, or HIGH) for a single
        customer message. Call this when the user asks how urgent a message,
        ticket, or conversation is.
      input_schema:
        type: object
        properties:
          msg: { type: string, description: The customer message text }
        required: [msg]
tool_resources:
  classify_escalation:
    identifier: FIELD_CX_DEMO.AI_FUNCTIONS.CLASSIFY_ESCALATION
    type: function
```

Then invoke the agent - the LLM decides to call the tool when the question warrants it:

```sql
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'FIELD_CX_DEMO.AI_FUNCTIONS.MY_CX_AGENT',
  { 'messages': [ { 'role': 'user',
      'content': [ { 'type': 'text',
        'text': 'How urgent is this message: "I have emailed five times and still no refund - cancelling today."' } ] } ] }
);
```

> The agent calls the UDF as a tool; the UDF runs `AI_COMPLETE` internally. You are billed for both the agent's orchestration tokens and the UDF's inference tokens - factor tool-use into the cost estimates from the base lab's Section 9.

In [ ]:
%%sql -r agent_check
-- The agent is real (created by setup.sql). Confirm it exists, then chat with it in
-- Snowsight (AI & ML -> Agents -> "CX Intelligence"). It combines Cortex Analyst over
-- the semantic view, Cortex Search over the chat telemetry, and your UDF as a tool.
SHOW AGENTS LIKE 'CX_INTELLIGENCE_AGENT' IN SCHEMA FIELD_CX_DEMO.ANALYTICS;

---
## Extension 2 - AI-function UDFs inside Cortex Analyst

Cortex Analyst answers natural-language questions by generating SQL over a **semantic view**. If a dimension in that view is backed by an AI-function UDF, Analyst's generated SQL transparently invokes the model. That means a business user can ask *"how many high-urgency chats came in this week?"* and the urgency label is computed by your UDF on the fly.

Expose the UDF as a **computed column** in a view, then build the semantic view on top of it. (Materialize the column into a table if you want to avoid re-running inference on every query - see the cost note below.)

In [ ]:
%%sql -r create_escalation_view
-- Computed-column view: escalation_level is produced by the AI-function UDF.
CREATE OR REPLACE VIEW CHAT_ESCALATION_V AS
SELECT
    t.thread_id,
    t.customer_id,
    t.channel,
    t.created_at,
    t.transcript,
    CLASSIFY_ESCALATION(t.transcript) AS escalation_level
FROM CHAT_THREADS t;

### Point a semantic view at the computed column

In the semantic model, `escalation_level` is just another dimension - Analyst does not need to know it is AI-generated:

```yaml
tables:
  - name: chat_escalation
    base_table: FIELD_CX_DEMO.AI_FUNCTIONS.CHAT_ESCALATION_V
    dimensions:
      - name: escalation_level
        expr: escalation_level
        description: AI-derived urgency of the chat (LOW / MEDIUM / HIGH)
      - name: channel
        expr: channel
    facts:
      - name: thread_count
        expr: COUNT(thread_id)
```

A user asks Analyst *"count chats by escalation level this month"* and it generates:

```sql
SELECT escalation_level, COUNT(thread_id)
FROM CHAT_ESCALATION_V
WHERE created_at >= DATE_TRUNC('month', CURRENT_DATE())
GROUP BY escalation_level;
```

> **Cost note:** a view re-runs `CLASSIFY_ESCALATION` on every query. For anything interactive, **materialize** the labels (a table refreshed by a task, or a Dynamic Table) so inference runs once per row, not once per question. Use the base-lab estimator to price the initial backfill.

In [ ]:
%%sql -r sv_by_plan
-- Runs live against the real semantic view created by setup.sql. Governed metrics,
-- no JOINs: churn plus the app-feedback thumbs-down rate that the app UX telemetry
-- (base lab, Section 1b) fed into ANALYTICS.CUSTOMER_FEEDBACK.
SELECT * FROM SEMANTIC_VIEW(
    FIELD_CX_DEMO.ANALYTICS.CX_ANALYTICS_SV
    METRICS churn_rate, thumbs_down_rate, customer_count
    DIMENSIONS customers.plan
);

---
## Extension 3 - AI functions in a Cortex Search enrichment pipeline

Cortex Search indexes text for hybrid (vector + keyword) retrieval. AI functions are the **pre-processing** step that turns raw documents into clean, structured, enriched rows before indexing:

- `AI_EXTRACT` pulls structured fields (product area, requested action) out of free text.
- `AI_EMBED` can generate embeddings for custom similarity logic.
- `AI_CLASSIFY` / `AI_SENTIMENT` add filterable attributes the search service can use as columns.

We enrich a small slice of `SUPPORT_TICKETS` (kept to `LIMIT 25` so it runs quickly - AI functions can be slow on a cold model).

In [ ]:
%%sql -r build_enriched
-- Enrichment pipeline: structured fields + embedding for a slice of tickets.
-- Scale the LIMIT up once you have priced the full run (base-lab Section 9).
CREATE OR REPLACE TABLE SUPPORT_TICKETS_ENRICHED AS
SELECT
    ticket_id,
    customer_id,
    subject,
    body,
    AI_EXTRACT(body, ['product_area','requested_action']) AS extracted,
    AI_EMBED('snowflake-arctic-embed-m-v1.5', body)        AS body_vec
FROM SUPPORT_TICKETS
LIMIT 25;

In [ ]:
%%sql -r enriched_preview
-- Flatten the AI_EXTRACT JSON into columns the search service can index/filter on.
SELECT
    ticket_id,
    subject,
    extracted:response:product_area::STRING    AS product_area,
    extracted:response:requested_action::STRING AS requested_action
FROM SUPPORT_TICKETS_ENRICHED
LIMIT 10;

### Index the enriched rows into Cortex Search

With the enriched table in place, the search service indexes `body` for retrieval and carries the AI-extracted fields as filter attributes:

```sql
CREATE OR REPLACE CORTEX SEARCH SERVICE support_ticket_search
  ON body
  ATTRIBUTES product_area, requested_action, customer_id
  WAREHOUSE = CX_AI_FUNCTIONS_WH
  TARGET_LAG = '1 hour'
  AS (
    SELECT
      ticket_id,
      body,
      extracted:response:product_area::STRING    AS product_area,
      extracted:response:requested_action::STRING AS requested_action,
      customer_id
    FROM SUPPORT_TICKETS_ENRICHED
  );
```

> This closes the loop: the same AI functions that produce CX telemetry also feed retrieval. A Cortex Agent can then combine **Analyst** (metrics from the semantic view), **Search** (relevant tickets), and your **custom UDF tool** (urgency) in a single answer.

In [ ]:
%%sql -r chat_search
-- Runs live against the real CHAT_SEARCH service created by setup.sql.
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'FIELD_CX_DEMO.ANALYTICS.CHAT_SEARCH',
        '{ "query": "customer wants to cancel or get a refund",
           "columns": ["transcript", "customer_id"],
           "limit": 5 }'
    )
)['results'] AS results;

---
## Part 2 - Best practices & cost/spike prevention

AI functions bill by **tokens processed**, so cost control is mostly about **how many tokens** you send and **which model** processes them.

### Prompt engineering
- Keep system prompts **specific and bounded** - vague prompts produce unpredictable output lengths and higher output-token cost.
- For classification, **list the valid labels** and instruct the model to answer in one word (as `CLASSIFY_ESCALATION` does).
- Use **structured output** (`responseFormat` for `AI_EXTRACT`, response schemas for `AI_COMPLETE`) to bound output size.

### Model selection for cost/quality tradeoff
- Smaller models (`llama3.1-8b`) cost far less per token than frontier models (`claude-sonnet-4`). The base-lab estimator showed a ~13x swing on the same data.
- For bounded classification, a **small model + tuned prompt** often matches a large one. Prove it empirically with the **AI Function Studio Optimize** workflow (base lab, Section 8), which plots an accuracy-vs-cost Pareto across models.

### Controlling cost spikes (reference SQL)

These run as an admin (`ACCOUNTADMIN` or a delegated role). Prefer **per-user quotas** (base lab, Section 9) as the primary hard cap; the patterns below are the belt-and-suspenders layer and support custom logic.

**1. Account / monthly spend alert** (Task + Alert on usage history):
```sql
CREATE OR REPLACE ALERT ai_monthly_spend_alert
  WAREHOUSE = CX_AI_FUNCTIONS_WH
  SCHEDULE = '60 MINUTE'
  IF (EXISTS (
    SELECT 1
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
    WHERE start_time >= DATE_TRUNC('month', CURRENT_TIMESTAMP())
    HAVING SUM(credits) > 500        -- monthly credit budget
  ))
  THEN CALL SYSTEM$SEND_EMAIL('cx_admin_ni', 'admin@example.com',
         'AI Function spend over budget', 'Monthly AI credits exceeded 500.');
ALTER ALERT ai_monthly_spend_alert RESUME;
```

**2. Per-user spending limit** - revoke access when a user exceeds their monthly budget (hourly Task):
```sql
CREATE OR REPLACE TASK enforce_user_budget
  WAREHOUSE = CX_AI_FUNCTIONS_WH SCHEDULE = '60 MINUTE' AS
  EXECUTE IMMEDIATE $$
  BEGIN
    LET over CURSOR FOR
      SELECT u.name
      FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY h
      JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON h.user_id = u.user_id
      WHERE h.start_time >= DATE_TRUNC('month', CURRENT_TIMESTAMP())
      GROUP BY u.name HAVING SUM(h.credits) > 120;
    FOR r IN over DO
      EXECUTE IMMEDIATE 'REVOKE ROLE ai_functions_user_role FROM USER "' || r.name || '"';
    END FOR;
  END; $$;
```

**3. Runaway query cancellation** - cancel in-flight queries past a credit threshold:
```sql
SELECT SYSTEM$CANCEL_QUERY('<query_id>');   -- see base-lab cell that lists still-running AI queries
```

**4. Query tagging for attribution** - populates QUERY_TAG in the usage history for team-level chargeback:
```sql
ALTER SESSION SET QUERY_TAG = 'team:cx-analytics;project:telemetry';
```

### Access control

Gate AI functions behind a dedicated role so quotas and limits cannot be bypassed:

| Database role | Grants |
|---|---|
| `SNOWFLAKE.CORTEX_USER` | All AI functions (granted to `PUBLIC` by default) |
| `SNOWFLAKE.AI_FUNCTIONS_USER` | Scalar AI functions only |
| `SNOWFLAKE.CORTEX_EMBED_USER` | `AI_EMBED` only |

```sql
USE ROLE ACCOUNTADMIN;
-- Remove the blanket grant, then hand access out only through a managed role
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;
CREATE ROLE IF NOT EXISTS ai_functions_user_role;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE ai_functions_user_role;
```

The account privilege **`USE AI FUNCTIONS`** must also be enabled at the account level for any of these to work.

---
## Summary

**Extensions** - the same AI-function UDFs from the base lab plug into the wider Cortex stack:
- **Agent tool** - `CLASSIFY_ESCALATION` registered as a generic tool the agent calls via tool-use.
- **Cortex Analyst** - the UDF as a computed column in a semantic view, so NL questions compute AI labels on the fly (materialize for interactive use).
- **Cortex Search** - `AI_EXTRACT` / `AI_EMBED` enrich documents before indexing, and expose filterable attributes.

**Cost/spike prevention** - bounded prompts, right-sized models, per-user quotas as the hard cap, plus spend alerts, budget-enforcement tasks, runaway cancellation, query tagging, and role-gated access.

Estimate first (base lab, Section 9), pick the smallest model that passes an eval, and cap with quotas - then scale the `LIMIT` off the enrichment and telemetry pipelines with confidence.